# 13 — Generation and KV Cache

## Goal

A trained decoder-only Transformer predicts a distribution over the next
token.

During training, the entire target sequence is already available, so the
model can process all sequence positions in parallel.

Autoregressive generation is different.

The next token does not exist until it has been sampled, so generation
must proceed sequentially:

$$
x_1
\rightarrow
x_2
\rightarrow
x_3
\rightarrow
\cdots
$$

This lesson studies:

- autoregressive generation,
- greedy decoding,
- temperature,
- top-k sampling,
- top-p sampling,
- the computational inefficiency of naive decoding,
- and the key-value cache used to avoid repeated attention computation.

In [1]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

from llmfp.nn import (
    SwiGLU,
    apply_rope,
    build_rope_cos_sin,
)

## 1. Autoregressive Generation

Given a prompt with shape

$$
(B,T),
$$

the model produces logits

$$
(B,T,V).
$$

During generation, only the logits from the final sequence position are
needed:

$$
(B,T,V)
\rightarrow
(B,V).
$$

These logits represent the model's prediction for the token that should
come after the current sequence.

After selecting one token, that token is appended to the sequence and
the model is called again.

In [2]:
def generate_greedy(
    model: torch.nn.Module, input_ids: torch.Tensor, max_new_tokens: int
) -> torch.Tensor:
    """Generate tokens by repeatedly selecting the highest-logit token.

    Args:
        model: Decoder-only language model returning logits of shape
            `(B, T, V)`.
        input_ids: Initial token IDs with shape `(B, T)`.
        max_new_tokens: Number of new tokens to generate.

    Returns:
        Token IDs containing the original prompt followed by generated
        tokens.
    """
    generated: torch.Tensor = input_ids

    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Recompute logits for the entire sequence generated so far.
            logits: torch.Tensor = model(generated)

            # Only the final position predicts the next token.
            next_token_logits: torch.Tensor = logits[:, -1, :]

            # Greedy decodings selects the token with the largest logits
            next_token: torch.Tensor = torch.argmax(
                next_token_logits, dim=-1, keepdim=True
            )

            # Append the selected token to the running sequence.
            generated = torch.cat([generated, next_token], dim=1)
    return generated

In [3]:
# sampled decoding
def generate_sampled(
    model: torch.nn.Module, input_ids: torch.Tensor, max_new_tokens: int
) -> torch.Tensor:
    """Generate tokens by sampling from the model distribution.

    Args:
        model: Decoder-only language model returning logits of shape
            `(B, T, V)`.
        input_ids: Initial token IDs with shape `(B, T)`.
        max_new_tokens: Number of new tokens to generate.

    Returns:
        Token IDs containing the prompt followed by sampled tokens.
    """
    generated: torch.Tensor = input_ids

    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits: torch.Tensor = model(generated)

            next_token_logits: torch.Tensor = logits[:, -1, :]

            # Convert raw logits into a categorical probability
            # distribution over the vocabulary.
            probabilities: torch.Tensor = F.softmax(next_token_logits, dim=-1)

            next_token: torch.Tensor = torch.multinomial(
                probabilities, num_samples=1
            )

            generated = torch.cat([generated, next_token], dim=1)

    return generated


## 2. Temperature Sampling

The model produces raw logits for the next token:

$$
z \in \mathbb{R}^{V}.
$$

Normally, probabilities are obtained with

$$
p_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}.
$$

Temperature modifies the logits before softmax:

$$
p_i(T)
=
\frac{
e^{z_i/T}
}{
\sum_j e^{z_j/T}
}.
$$

The temperature $T$ controls how concentrated the probability
distribution is.

- $T < 1$ makes the distribution sharper.
- $T = 1$ leaves the distribution unchanged.
- $T > 1$ makes the distribution flatter.

In [4]:
logits: torch.Tensor = torch.tensor([4.0, 2.0, 1.0])

for temperature in [0.5, 1.0, 2.0]:
    probabilities: torch.Tensor = F.softmax(
        logits / temperature,
        dim=-1,
    )

    print(
        f"T={temperature}:",
        probabilities,
    )

T=0.5: tensor([0.9796, 0.0179, 0.0024])
T=1.0: tensor([0.8438, 0.1142, 0.0420])
T=2.0: tensor([0.6285, 0.2312, 0.1402])


In [5]:
def sample_with_temperature(
    logits: torch.Tensor,
    temperature: float,
) -> torch.Tensor:
    """Sample token IDs from logits using temperature scaling.

    Args:
        logits: Next-token logits with shape `(B, V)`.
        temperature: Positive value controlling distribution sharpness.

    Returns:
        Sampled token IDs with shape `(B, 1)`.

    Raises:
        ValueError: If `temperature` is not positive.
    """
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero.")

    # Lower temperatures enlarge logit differences; higher
    # temperatures compress them.
    scaled_logits: torch.Tensor = logits / temperature

    probabilities: torch.Tensor = F.softmax(
        scaled_logits,
        dim=-1,
    )

    return torch.multinomial(
        probabilities,
        num_samples=1,
    )

In [6]:
def generate_with_temperature(
    model: torch.nn.Module,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float,
) -> torch.Tensor:
    """Generate tokens autoregressively using temperature sampling.

    Args:
        model: Decoder-only language model returning `(B, T, V)` logits.
        input_ids: Prompt token IDs with shape `(B, T)`.
        max_new_tokens: Number of tokens to generate.
        temperature: Positive sampling temperature.

    Returns:
        Prompt followed by generated token IDs.
    """
    generated: torch.Tensor = input_ids

    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Naive decoding recomputes the whole sequence every step.
            logits: torch.Tensor = model(generated)

            # Only the final position predicts the next token.
            next_token_logits: torch.Tensor = logits[:, -1, :]

            next_token: torch.Tensor = sample_with_temperature(
                next_token_logits,
                temperature,
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1,
            )

    return generated

### Temperature Intuition

Temperature does not change the ordering of logits.

Instead, it changes how strongly the model prefers high-logit tokens
over lower-logit alternatives.

<pre>
low temperature
    ↓
sharper distribution
    ↓
more concentrated sampling

high temperature
    ↓
flatter distribution
    ↓
more diverse sampling
</pre>

Temperature affects sampling only when tokens are actually sampled from
the distribution. Greedy decoding remains unchanged because the largest
logit keeps the same rank.

## 3. Top-k Sampling

Temperature rescales the entire vocabulary distribution.

Top-k sampling instead restricts the candidate set.

Given vocabulary logits

$$
z \in \mathbb{R}^{V},
$$

top-k sampling keeps only the $k$ largest logits and removes all other
tokens from consideration.

The remaining logits are then normalized with softmax and sampled.

Conceptually:

<pre>
all vocabulary tokens
        ↓
keep highest k logits
        ↓
softmax over surviving tokens
        ↓
sample
</pre>

Top-k therefore prevents very low-probability tokens from being sampled,
even when temperature makes the distribution relatively flat.

In [7]:
def top_k_filter(logits: torch.Tensor, k: int) -> torch.Tensor:
    """Keep only the k largest logits in each batch row.

    Args:
        logits: Next-token logits with shape `(B, V)`.
        k: Number of highest-logit tokens to keep.

    Returns:
        Filtered logits with the same shape `(B, V)`. Tokens outside
        the top-k set are replaced by negative infinity.

    Raises:
        ValueError: If `k` is not between 1 and the vocabulary size.
    """
    vocab_size: int = logits.shape[-1]

    if not 1 <= k <= vocab_size:
        raise ValueError("k must be between 1 and the vocab size")

    # Find the k largest logits for each batch example.

    top_values, _ = torch.topk(logits, k=k, dim=-1)

    # The smallest value among the retained logits is the cutoff.
    threshold: torch.Tensor = top_values[:, -1:]

    # Remove every token whose logit is below the cutoff.
    filtered_logits: torch.Tensor = logits.masked_fill(
        logits < threshold, float("-inf")
    )

    return filtered_logits

In [8]:
logits: torch.Tensor = torch.tensor([[5.0, 4.0, 3.0, 1.0, 0.0, -2.0]])

filtered_logits: torch.Tensor = top_k_filter(
    logits,
    k=3,
)

print("original:")
print(logits)

print("\nfiltered:")
print(filtered_logits)

original:
tensor([[ 5.,  4.,  3.,  1.,  0., -2.]])

filtered:
tensor([[5., 4., 3., -inf, -inf, -inf]])


In [9]:
probabilities: torch.Tensor = F.softmax(
    filtered_logits,
    dim=-1,
)

print(probabilities)

tensor([[0.6652, 0.2447, 0.0900, 0.0000, 0.0000, 0.0000]])


In [10]:
def sample_top_k(
    logits: torch.Tensor,
    temperature: float,
    k: int,
) -> torch.Tensor:
    """Sample token IDs using temperature and top-k filtering.

    Args:
        logits: Next-token logits with shape `(B, V)`.
        temperature: Positive value controlling distribution sharpness.
        k: Number of highest-logit tokens allowed to remain.

    Returns:
        Sampled token IDs with shape `(B, 1)`.

    Raises:
        ValueError: If `temperature` is not positive.
    """
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero.")

    # Temperature changes the relative spacing between logits.
    scaled_logits: torch.Tensor = logits / temperature

    # Top-k removes all but the k strongest candidates.
    filtered_logits: torch.Tensor = top_k_filter(
        scaled_logits,
        k=k,
    )

    probabilities: torch.Tensor = F.softmax(
        filtered_logits,
        dim=-1,
    )

    return torch.multinomial(
        probabilities,
        num_samples=1,
    )

In [11]:
def generate_top_k(
    model: torch.nn.Module,
    input_ids: torch.Tensor,
    max_new_tokens: int,
    temperature: float,
    k: int,
) -> torch.Tensor:
    """Generate tokens using temperature-scaled top-k sampling.

    Args:
        model: Decoder-only language model returning `(B, T, V)` logits.
        input_ids: Prompt token IDs with shape `(B, T)`.
        max_new_tokens: Number of tokens to generate.
        temperature: Positive sampling temperature.
        k: Number of candidate tokens retained at each generation step.

    Returns:
        Prompt followed by generated token IDs.
    """
    generated: torch.Tensor = input_ids

    model.eval()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Naive decoding recomputes all previous sequence positions.
            logits: torch.Tensor = model(generated)

            next_token_logits: torch.Tensor = logits[:, -1, :]

            next_token: torch.Tensor = sample_top_k(
                next_token_logits,
                temperature=temperature,
                k=k,
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1,
            )

    return generated

## 4. Top-p / Nucleus Sampling

Top-k sampling keeps a fixed number of candidate tokens.

Top-p sampling instead keeps the smallest set of high-probability tokens
whose cumulative probability reaches a threshold $p$.

For example, if the sorted next-token probabilities are

$$
[0.50,\ 0.25,\ 0.15,\ 0.05,\ 0.05],
$$

their cumulative probabilities are

$$
[0.50,\ 0.75,\ 0.90,\ 0.95,\ 1.00].
$$

With

$$
p=0.90,
$$

the first three tokens are sufficient to cover 90% of the probability
mass.

The number of surviving tokens therefore changes dynamically from one
generation step to another.

In [12]:
probabilities: torch.Tensor = torch.tensor([[0.50, 0.25, 0.15, 0.05, 0.05]])

cumulative_probabilities: torch.Tensor = torch.cumsum(
    probabilities,
    dim=-1,
)

print(cumulative_probabilities)

tensor([[0.5000, 0.7500, 0.9000, 0.9500, 1.0000]])


In [13]:
def top_p_filter(logits: torch.Tensor, p: float) -> torch.Tensor:
    """Keep the smallest high-probability token set covering mass p.

    Args:
        logits: Next-token logits with shape `(B, V)`.
        p: Cumulative probability threshold in the interval `(0, 1]`.

    Returns:
        Filtered logits with shape `(B, V)`. Tokens outside the nucleus
        are replaced by negative infinity.

    Raises:
        ValueError: If `p` is not in the interval `(0, 1]`.
    """
    if not 0.0 < p <= 1.0:
        raise ValueError("p must be greater than 0 and at most 1")

    # Sort vocab candidates from the hightest to lowest logit.
    sorted_logits, sorted_indices = torch.sort(logits, dim=-1, descending=True)

    # Compute probabilities in sorted order
    sorted_probabilities: torch.Tensor = F.softmax(sorted_logits, dim=-1)

    # Measure how much probability mass has been accumulated.
    cumulative_probabilities: torch.Tensor = torch.cumsum(
        sorted_probabilities, dim=-1
    )

    # Initially mark every token than lies beyond the threshold.
    sorted_remove_mask: torch.Tensor = cumulative_probabilities > p

    # Keep the first token that crosses the threshold so that the
    # retained set actually reaches at least probability mass p.
    sorted_remove_mask[:, 1:] = sorted_remove_mask[:, :-1].clone()

    sorted_remove_mask[:, 0] = False

    filtered_sorted_logits: torch.Tensor = sorted_logits.masked_fill(
        sorted_remove_mask, float("-inf")
    )

    # Restore logits to their original vocabulary order.
    filtered_logits: torch.Tensor = torch.full_like(logits, float("-inf"))

    filtered_logits.scatter_(
        dim=-1, index=sorted_indices, src=filtered_sorted_logits
    )

    return filtered_logits

In [14]:
logits: torch.Tensor = torch.tensor(
    [
        [
            1.0,
            5.0,
            2.0,
            4.0,
            3.0,
        ]
    ]
)

filtered_logits: torch.Tensor = top_p_filter(
    logits,
    p=0.8,
)

print("original:")
print(logits)

print("\nfiltered:")
print(filtered_logits)

print("\nprobabilities:")
print(
    F.softmax(
        filtered_logits,
        dim=-1,
    )
)

original:
tensor([[1., 5., 2., 4., 3.]])

filtered:
tensor([[-inf, 5., -inf, 4., -inf]])

probabilities:
tensor([[0.0000, 0.7311, 0.0000, 0.2689, 0.0000]])


In [15]:
def sample_top_p(
    logits: torch.Tensor,
    temperature: float,
    p: float,
) -> torch.Tensor:
    """Sample token IDs using temperature-scaled top-p sampling.

    Args:
        logits: Next-token logits with shape `(B, V)`.
        temperature: Positive value controlling distribution sharpness.
        p: Cumulative probability threshold in `(0, 1]`.

    Returns:
        Sampled token IDs with shape `(B, 1)`.

    Raises:
        ValueError: If `temperature` is not positive.
    """
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero.")

    # Temperature is applied before top-p because it changes the
    # probability distribution and therefore changes the nucleus.
    scaled_logits: torch.Tensor = logits / temperature

    filtered_logits: torch.Tensor = top_p_filter(
        scaled_logits,
        p=p,
    )

    probabilities: torch.Tensor = F.softmax(
        filtered_logits,
        dim=-1,
    )

    return torch.multinomial(
        probabilities,
        num_samples=1,
    )

### Top-k vs Top-p

Top-k uses a fixed candidate count:

$$
k = \text{constant}.
$$

Top-p uses a fixed probability-mass threshold:

$$
p = \text{constant},
$$

while the number of surviving tokens changes dynamically.

<pre>
Top-k
    → fixed number of candidates

Top-p
    → adaptive number of candidates
    → fixed cumulative probability mass
</pre>

Temperature and top-p interact because temperature changes the
probability distribution used to construct the nucleus.

## 5. Why Naive Generation Is Wasteful

Naive autoregressive generation runs the model on the entire sequence
after every newly generated token.

Suppose the prompt is

<pre>
A B C
</pre>

The first generation step processes

<pre>
A B C
</pre>

and generates `D`.

The next step processes

<pre>
A B C D
</pre>

again, even though the model has already computed representations for
`A`, `B`, and `C`.

After generating `E`, the model processes

<pre>
A B C D E
</pre>

again.

This repeatedly recomputes work for tokens whose representations cannot
change because causal attention prevents future tokens from influencing
earlier positions.

### Naive Decoding vs Cached Decoding

Without a KV cache:

<pre>
step 1:  A B C
step 2:  A B C D
step 3:  A B C D E
              ↑
      old tokens are recomputed
</pre>

With a KV cache:

<pre>
prefill: A B C
         ↓
         cache K_A K_B K_C
         cache V_A V_B V_C

decode D:
         compute Q_D K_D V_D
         append K_D V_D to cache

decode E:
         compute Q_E K_E V_E
         append K_E V_E to cache
</pre>

After the initial prompt has been processed, each decoding step only
needs to process the newly generated token.

## 6. Position Offsets During Cached Decoding

During naive generation, the complete sequence is passed through the
model at every step, so token positions can always be generated from

$$
0,\ldots,T-1.
$$

Cached decoding is different.

After the prompt has been processed, each decoding step may contain only
one new token:

$$
(B,1).
$$

However, that token is not at position zero.

If the KV cache already contains $T_{\text{past}}$ tokens, the new token
has absolute position

$$
T_{\text{past}}.
$$

RoPE must therefore use the cache length as a positional offset.

In [16]:
def rope_frequencies(
    head_dim: int,
    base: float = 10000.0,
    *,
    device: torch.device | None = None,
) -> torch.Tensor:
    """Compute the angular frequencies used by Rotary Position Embeddings.

    Each adjacent pair of attention-head features shares one frequency.

    Args:
        head_dim: Feature dimension of one attention head.
        base: Base controlling the geometric spacing of RoPE frequencies.
        device: Device on which to create the frequency tensor.

    Returns:
        A tensor of shape `(head_dim / 2,)`, containing one frequency
        for each adjacent feature pair.

    Raises:
        ValueError: If `head_dim` is not even.
    """
    if head_dim % 2 != 0:
        raise ValueError("head_dim must be even for RoPE.")

    # One frequency is used for each adjacent feature pair:
    # (0, 1), (2, 3), (4, 5), ...
    dimension_indices: torch.Tensor = torch.arange(
        0,
        head_dim,
        2,
        dtype=torch.float32,
        device=device,
    )

    frequencies: torch.Tensor = torch.pow(
        base,
        -dimension_indices / head_dim,
    )

    return frequencies


def build_rope_cache(
    sequence_length: int,
    head_dim: int,
    *,
    position_offset: int = 0,
    base: float = 10000.0,
    device: torch.device | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Build RoPE cosine and sine values for a sequence range.

    Args:
        sequence_length: Number of new token positions.
        head_dim: Feature dimension of one attention head.
        position_offset: Absolute position of the first new token.
        base: Base controlling the RoPE frequency range.
        device: Device on which to create the tensors.

    Returns:
        Cosine and sine tensors with shape
        `(sequence_length, head_dim / 2)`.
    """
    frequencies: torch.Tensor = rope_frequencies(
        head_dim=head_dim,
        base=base,
        device=device,
    )

    # Cached decoding may start at a non-zero absolute position.
    positions: torch.Tensor = torch.arange(
        position_offset,
        position_offset + sequence_length,
        dtype=torch.float32,
        device=device,
    )

    # One angle is created for every (position, frequency) pair.
    angles: torch.Tensor = positions.unsqueeze(1) * frequencies.unsqueeze(0)

    return (
        torch.cos(angles),
        torch.sin(angles),
    )

In [17]:
cos_values, sin_values = build_rope_cache(
    sequence_length=3,
    head_dim=8,
    position_offset=0,
)

## 7. Cached Self-Attention

During cached decoding, the attention layer receives only the newly
generated token together with previously cached keys and values.

For a single decoding step,

$$
X_{\text{new}}
\in
\mathbb{R}^{B \times 1 \times C}.
$$

The layer computes

$$
Q_{\text{new}},
K_{\text{new}},
V_{\text{new}}
\in
\mathbb{R}^{B \times H \times 1 \times D}.
$$

The new key and value are appended to the existing cache:

$$
K_{\text{all}}
=
[K_{\text{past}};K_{\text{new}}],
$$

$$
V_{\text{all}}
=
[V_{\text{past}};V_{\text{new}}].
$$

The current query then attends to all cached keys and values:

$$
Q_{\text{new}}
K_{\text{all}}^\top.
$$

Only keys and values need to persist across decoding steps.

In [18]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with RoPE and optional KV caching.

    During ordinary training, the full sequence is processed at once and
    the module returns only the attention output.

    During autoregressive inference, the module can return and reuse
    cached keys and values so that previous tokens do not need to be
    projected again.

    Args:
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads.
        rope_base: Base controlling the RoPE frequency range.
    """

    def __init__(
        self,
        embedding_dim: int,
        num_heads: int,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if embedding_dim % num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")

        self.embedding_dim: int = embedding_dim
        self.num_heads: int = num_heads
        self.head_dim: int = embedding_dim // num_heads
        self.rope_base: float = rope_base

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")

        self.query_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

        self.key_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

        self.value_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

        self.output_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        past_key: torch.Tensor | None = None,
        past_value: torch.Tensor | None = None,
        use_cache: bool = False,
    ) -> (
        torch.Tensor
        | tuple[
            torch.Tensor,
            torch.Tensor,
            torch.Tensor,
        ]
    ):
        """Apply causal self-attention with optional KV caching.

        Args:
            x: New input representations with shape `(B, T_new, C)`.
            past_key: Previously cached keys with shape
                `(B, H, T_past, D)`, or `None`.
            past_value: Previously cached values with shape
                `(B, H, T_past, D)`, or `None`.
            use_cache: Whether to return the updated key and value cache.

        Returns:
            If `use_cache` is False:
                Attention output with shape `(B, T_new, C)`.

            If `use_cache` is True:
                A tuple containing:
                - attention output with shape `(B, T_new, C)`,
                - key cache with shape `(B, H, T_total, D)`,
                - value cache with shape `(B, H, T_total, D)`.

        Raises:
            ValueError: If input or cache shapes are inconsistent.
            ValueError: If cached decoding receives more than one new
                token in this educational implementation.
        """
        if x.ndim != 3:
            raise ValueError("x must have shape (B, T, C).")

        batch_size, sequence_length, embedding_dim = x.shape

        if embedding_dim != self.embedding_dim:
            raise ValueError(
                "The final input dimension must match embedding_dim."
            )

        # A valid cache must contain both K and V, or neither.
        if (past_key is None) != (past_value is None):
            raise ValueError(
                "past_key and past_value must either both be provided "
                "or both be None."
            )

        if past_key is not None:
            if not use_cache:
                raise ValueError(
                    "use_cache must be True when a past KV cache is provided."
                )

            if past_key.ndim != 4:
                raise ValueError("past_key must have shape (B, H, T_past, D).")

            if past_value is None:
                raise ValueError("past_value must be provided with past_key.")

            if past_value.shape != past_key.shape:
                raise ValueError(
                    "past_key and past_value must have identical shapes."
                )

            if past_key.shape[0] != batch_size:
                raise ValueError(
                    "The cache batch size must match the input batch size."
                )

            if past_key.shape[1] != self.num_heads:
                raise ValueError(
                    "The cache must use the same number of attention heads."
                )

            if past_key.shape[-1] != self.head_dim:
                raise ValueError(
                    "The cache head dimension does not match the model."
                )

            # For now, cached decoding processes exactly one new token.
            if sequence_length != 1:
                raise ValueError(
                    "Cached decoding currently expects exactly one new token."
                )

            past_length: int = past_key.shape[2]

        else:
            past_length = 0

        # Project only the tokens supplied in the current forward call.
        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # Split model width C into H attention heads of width D.
        #
        # (B, T_new, C)
        #     ->
        # (B, H, T_new, D)
        q = rearrange(
            q,
            "b t (h d) -> b h t d",
            h=self.num_heads,
        )

        k = rearrange(
            k,
            "b t (h d) -> b h t d",
            h=self.num_heads,
        )

        v = rearrange(
            v,
            "b t (h d) -> b h t d",
            h=self.num_heads,
        )

        # Cached decoding starts after all positions already present
        # in the KV cache.
        cos_values, sin_values = build_rope_cache(
            sequence_length=sequence_length,
            head_dim=self.head_dim,
            position_offset=past_length,
            base=self.rope_base,
            device=x.device,
        )

        # RoPE changes the positional geometry of Q and K.
        # V remains unchanged.
        q = apply_rope(
            q,
            cos_values,
            sin_values,
        )

        k = apply_rope(
            k,
            cos_values,
            sin_values,
        )

        if past_key is None:
            # During training or prompt prefill, the current sequence
            # supplies the complete K/V context.
            key_cache: torch.Tensor = k
            value_cache: torch.Tensor = v

        else:
            # Append only the newly computed K/V along the sequence axis.
            #
            # (B, H, T_past, D)
            #          +
            # (B, H, 1, D)
            #          ->
            # (B, H, T_past + 1, D)
            key_cache = torch.cat(
                [past_key, k],
                dim=2,
            )

            value_cache = torch.cat(
                [past_value, v],
                dim=2,
            )

        if past_key is None:
            # During full-sequence processing, multiple query positions
            # exist simultaneously, so a causal mask is required.
            is_causal: bool = True
        else:
            # During one-token decoding, every key in the cache is
            # already part of the current token's causal past.
            is_causal = False

        # SDPA performs:
        #
        # QK^T / sqrt(D)
        # -> causal masking when required
        # -> softmax
        # -> weighted sum over V.
        attended: torch.Tensor = F.scaled_dot_product_attention(
            q,
            key_cache,
            value_cache,
            is_causal=is_causal,
        )

        # Merge H heads back into the residual-stream width C.
        #
        # (B, H, T_new, D)
        #     ->
        # (B, T_new, C)
        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        output: torch.Tensor = self.output_projection(attended)

        if use_cache:
            return (
                output,
                key_cache,
                value_cache,
            )

        return output

## 8. Verifying Cached Attention

A KV cache is an optimization.

It should change **how much computation is repeated**, but it should not
change the mathematical result of attention.

Suppose the complete sequence contains four token representations:

<pre>
A B C D
</pre>

We can compute the output for `D` in two ways.

### Full-sequence computation

Process all four positions together:

<pre>
A B C D
      ↓
causal attention
      ↓
output for D
</pre>

### Cached computation

First prefill:

<pre>
A B C
  ↓
cache K/V
</pre>

Then process only:

<pre>
D
</pre>

using the cached keys and values from `A B C`.

If the KV-cache implementation is correct, both methods should produce
the same output for position `D`, up to floating-point error.

In [19]:
embedding_dim: int = 32
num_heads: int = 4

attention = CausalSelfAttention(
    embedding_dim=embedding_dim,
    num_heads=num_heads,
)

attention.eval()

x_full: torch.Tensor = torch.randn(
    1,
    4,
    embedding_dim,
)

print(x_full.shape)

torch.Size([1, 4, 32])


In [20]:
with torch.no_grad():
    full_output: torch.Tensor = attention(x_full)

print(full_output.shape)

torch.Size([1, 4, 32])


In [21]:
full_last_output: torch.Tensor = full_output[:, -1:, :]

print(full_last_output.shape)

torch.Size([1, 1, 32])


In [22]:
# prefill A, B, C
x_prompt: torch.Tensor = x_full[:, :3, :]

print(x_prompt.shape)

with torch.no_grad():
    (
        prompt_output,
        key_cache,
        value_cache,
    ) = attention(
        x_prompt,
        use_cache=True,
    )

print(
    "prompt output:",
    prompt_output.shape,
)

print(
    "key cache:",
    key_cache.shape,
)

print(
    "value cache:",
    value_cache.shape,
)

torch.Size([1, 3, 32])
prompt output: torch.Size([1, 3, 32])
key cache: torch.Size([1, 4, 3, 8])
value cache: torch.Size([1, 4, 3, 8])


In [23]:
# Only input D
x_new: torch.Tensor = x_full[:, 3:4, :]

print(x_new.shape)

with torch.no_grad():
    (
        cached_output,
        updated_key_cache,
        updated_value_cache,
    ) = attention(
        x_new,
        past_key=key_cache,
        past_value=value_cache,
        use_cache=True,
    )

print(
    "cached output:",
    cached_output.shape,
)

print(
    "updated key cache:",
    updated_key_cache.shape,
)

print(
    "updated value cache:",
    updated_value_cache.shape,
)

torch.Size([1, 1, 32])
cached output: torch.Size([1, 1, 32])
updated key cache: torch.Size([1, 4, 4, 8])
updated value cache: torch.Size([1, 4, 4, 8])


In [24]:
outputs_match: bool = torch.allclose(
    full_last_output,
    cached_output,
    atol=1e-5,
)

print(
    "cached output matches full output:",
    outputs_match,
)

cached output matches full output: True


### Why This Test Matters

Matching outputs verify more than the cache concatenation itself.

The comparison simultaneously checks that:

- cached keys and values are identical to their full-forward versions,
- new RoPE positions use the correct absolute offset,
- the current query sees exactly the same historical context,
- and cached decoding preserves the mathematical result of attention.

A KV cache is therefore a computational optimization, not a different
attention algorithm.

## 9. Propagating the KV Cache Through Transformer Blocks

Each Transformer layer computes its own keys and values.

Therefore, a multi-layer Transformer does not have one global KV cache.

Instead, every layer maintains its own pair:

$$
(K^{(l)}, V^{(l)}).
$$

For a model with $N$ layers, the complete cache can be represented as

$$
[
(K^{(0)},V^{(0)}),
(K^{(1)},V^{(1)}),
\ldots,
(K^{(N-1)},V^{(N-1)})
].
$$

During decoding, each Transformer block receives the cache belonging to
that layer and returns its updated version.

In [25]:
LayerKVCache = tuple[
    torch.Tensor,
    torch.Tensor,
]

KVCache = list[LayerKVCache]

In [26]:
# Change the `TransformerBlock`


class TransformerBlock(nn.Module):
    """A pre-norm decoder-only Transformer block.

    Args:
        embedding_dim: Width of the residual stream.
        attention: Causal self-attention module.
        feed_forward: Token-wise feed-forward module.
    """

    def __init__(
        self,
        embedding_dim: int,
        attention: nn.Module,
        feed_forward: nn.Module,
    ) -> None:
        super().__init__()

        self.attention_norm = nn.RMSNorm(embedding_dim)

        self.attention = attention

        self.feed_forward_norm = nn.RMSNorm(embedding_dim)

        self.feed_forward = feed_forward

    def forward(
        self,
        x: torch.Tensor,
        past_key: torch.Tensor | None = None,
        past_value: torch.Tensor | None = None,
        use_cache: bool = False,
    ) -> (
        torch.Tensor
        | tuple[
            torch.Tensor,
            torch.Tensor,
            torch.Tensor,
        ]
    ):
        """Apply one Transformer block with optional KV caching.

        Args:
            x: Residual-stream tensor with shape `(B, T_new, C)`.
            past_key: Cached keys for this layer, or `None`.
            past_value: Cached values for this layer, or `None`.
            use_cache: Whether updated K/V tensors should be returned.

        Returns:
            If `use_cache` is False:
                Updated residual stream `(B, T_new, C)`.

            If `use_cache` is True:
                `(x, key_cache, value_cache)`.
        """
        normalized: torch.Tensor = self.attention_norm(x)

        attention_result = self.attention(
            normalized,
            past_key=past_key,
            past_value=past_value,
            use_cache=use_cache,
        )

        if use_cache:
            (
                attention_output,
                key_cache,
                value_cache,
            ) = attention_result
        else:
            attention_output = attention_result

        # Attention contributes an update to the residual stream.
        x = x + attention_output

        # The feed-forward network remains token-wise and requires no cache.
        x = x + self.feed_forward(self.feed_forward_norm(x))

        if use_cache:
            return (
                x,
                key_cache,
                value_cache,
            )

        return x

## 10. Model-Level KV Cache

The GPT model coordinates one KV cache per Transformer layer.

During prefill, every block creates its initial cache.

During decoding, the cache for layer $l$ is passed only to block $l$.

The updated cache is collected and returned by the model.

The residual stream still flows sequentially through the Transformer
layers, while the KV cache provides persistent attention state for each
layer.

In [27]:
@dataclass
class GPTConfig:
    """Configuration for the decoder-only Transformer.

    Attributes:
        vocab_size: Number of tokens in the vocabulary.
        embedding_dim: Width of the residual stream.
        num_heads: Number of attention heads per Transformer block.
        num_layers: Number of stacked Transformer blocks.
        hidden_dim: Intermediate width of each SwiGLU network.
        rope_base: Base controlling the RoPE frequency range.
    """

    vocab_size: int
    embedding_dim: int
    num_heads: int
    num_layers: int
    hidden_dim: int
    rope_base: float = 10000.0

    def __post_init__(self) -> None:
        """Validate architectural relationships."""
        if self.vocab_size <= 0:
            raise ValueError("vocab_size must be positive.")

        if self.embedding_dim <= 0:
            raise ValueError("embedding_dim must be positive.")

        if self.num_heads <= 0:
            raise ValueError("num_heads must be positive.")

        if self.num_layers <= 0:
            raise ValueError("num_layers must be positive.")

        if self.hidden_dim <= 0:
            raise ValueError("hidden_dim must be positive.")

        if self.embedding_dim % self.num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")

        head_dim: int = self.embedding_dim // self.num_heads

        if head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")

In [28]:
class GPT(nn.Module):
    """Decoder-only Transformer language model with optional KV caching.

    Args:
        config: Model architecture configuration.
    """

    def __init__(
        self,
        config: GPTConfig,
    ) -> None:
        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.embedding_dim,
        )

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    embedding_dim=config.embedding_dim,
                    attention=CausalSelfAttention(
                        embedding_dim=config.embedding_dim,
                        num_heads=config.num_heads,
                        rope_base=config.rope_base,
                    ),
                    feed_forward=SwiGLU(
                        embedding_dim=config.embedding_dim,
                        hidden_dim=config.hidden_dim,
                    ),
                )
                for _ in range(config.num_layers)
            ]
        )

        self.final_norm = nn.RMSNorm(config.embedding_dim)

        self.lm_head = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False,
        )

    def forward(
        self,
        token_ids: torch.Tensor,
        past_key_values: KVCache | None = None,
        use_cache: bool = False,
    ) -> (
        torch.Tensor
        | tuple[
            torch.Tensor,
            KVCache,
        ]
    ):
        """Convert token IDs into logits with optional KV caching.

        Args:
            token_ids: New token IDs with shape `(B, T_new)`.
            past_key_values: One `(K, V)` cache pair per Transformer
                layer, or `None`.
            use_cache: Whether updated KV caches should be returned.

        Returns:
            If `use_cache` is False:
                Vocabulary logits with shape `(B, T_new, V)`.

            If `use_cache` is True:
                `(logits, new_cache)` where `new_cache` contains one
                `(K, V)` pair for each Transformer block.

        Raises:
            ValueError: If token IDs or cache structure are invalid.
        """
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape (B, T).")

        if past_key_values is not None:
            if not use_cache:
                raise ValueError(
                    "use_cache must be True when past_key_values are provided."
                )

            if len(past_key_values) != len(self.blocks):
                raise ValueError(
                    "past_key_values must contain exactly one "
                    "cache pair per Transformer block."
                )

        # Convert discrete token IDs into residual-stream vectors.
        #
        # (B, T_new)
        #     ->
        # (B, T_new, C)
        x: torch.Tensor = self.token_embedding(token_ids)

        new_cache: KVCache = []

        for layer_index, block in enumerate(self.blocks):
            if past_key_values is None:
                past_key = None
                past_value = None
            else:
                (
                    past_key,
                    past_value,
                ) = past_key_values[layer_index]

            block_result = block(
                x,
                past_key=past_key,
                past_value=past_value,
                use_cache=use_cache,
            )

            if use_cache:
                if not isinstance(
                    block_result,
                    tuple,
                ):
                    raise RuntimeError(
                        "Transformer block did not return a KV cache."
                    )

                (
                    x,
                    key_cache,
                    value_cache,
                ) = block_result

                # Cache order matches Transformer layer order.
                new_cache.append(
                    (
                        key_cache,
                        value_cache,
                    )
                )

            else:
                if isinstance(
                    block_result,
                    tuple,
                ):
                    raise RuntimeError("Unexpected KV cache returned.")

                x = block_result

        # Normalize the final residual-stream representation.
        x = self.final_norm(x)

        # Produce vocabulary logits for every supplied token position.
        #
        # (B, T_new, C)
        #      ->
        # (B, T_new, V)
        logits: torch.Tensor = self.lm_head(x)

        if use_cache:
            return (
                logits,
                new_cache,
            )

        return logits

In [29]:
config = GPTConfig(
    vocab_size=100,
    embedding_dim=32,
    num_heads=4,
    num_layers=3,
    hidden_dim=96,
)

model = GPT(config)

token_ids: torch.Tensor = torch.randint(
    0,
    config.vocab_size,
    (2, 10),
)

logits: torch.Tensor = model(token_ids)

print(logits.shape)

torch.Size([2, 10, 100])


In [30]:
prompt_ids: torch.Tensor = torch.tensor([[5, 8, 2, 9]])

model.eval()

with torch.no_grad():
    prefill_result = model(
        prompt_ids,
        use_cache=True,
    )

if not isinstance(
    prefill_result,
    tuple,
):
    raise RuntimeError("Expected cached model output.")

prompt_logits, cache = prefill_result

print(
    "prompt logits:",
    prompt_logits.shape,
)

print(
    "cached layers:",
    len(cache),
)

prompt logits: torch.Size([1, 4, 100])
cached layers: 3


In [31]:
for layer_index, (
    key_cache,
    value_cache,
) in enumerate(cache):
    print(
        f"layer {layer_index}",
        "K:",
        key_cache.shape,
        "V:",
        value_cache.shape,
    )

layer 0 K: torch.Size([1, 4, 4, 8]) V: torch.Size([1, 4, 4, 8])
layer 1 K: torch.Size([1, 4, 4, 8]) V: torch.Size([1, 4, 4, 8])
layer 2 K: torch.Size([1, 4, 4, 8]) V: torch.Size([1, 4, 4, 8])


In [32]:
next_token: torch.Tensor = torch.tensor([[17]])

with torch.no_grad():
    decode_result = model(
        next_token,
        past_key_values=cache,
        use_cache=True,
    )

if not isinstance(
    decode_result,
    tuple,
):
    raise RuntimeError("Expected cached model output.")

new_logits, cache = decode_result

print(new_logits.shape)

torch.Size([1, 1, 100])


## 11. Verifying the Full GPT KV Cache

A KV cache should change computation, not model predictions.

Consider the sequence

<pre>
A B C D
</pre>

There are two ways to compute the logits produced after token `D`.

### Full forward

Process the entire sequence:

<pre>
A B C D
      ↓
complete GPT
      ↓
last-position logits
</pre>

### Cached forward

First prefill:

<pre>
A B C
  ↓
create one KV cache per Transformer layer
</pre>

Then process only:

<pre>
D
</pre>

using the cached keys and values.

Both methods should produce the same logits for the position
corresponding to `D`, up to floating-point error.

In [33]:
config = GPTConfig(
    vocab_size=100,
    embedding_dim=32,
    num_heads=4,
    num_layers=3,
    hidden_dim=96,
)

model = GPT(config)

model.eval()

GPT(
  (token_embedding): Embedding(100, 32)
  (blocks): ModuleList(
    (0-2): 3 x TransformerBlock(
      (attention_norm): RMSNorm((32,), eps=None, elementwise_affine=True)
      (attention): CausalSelfAttention(
        (query_projection): Linear(in_features=32, out_features=32, bias=False)
        (key_projection): Linear(in_features=32, out_features=32, bias=False)
        (value_projection): Linear(in_features=32, out_features=32, bias=False)
        (output_projection): Linear(in_features=32, out_features=32, bias=False)
      )
      (feed_forward_norm): RMSNorm((32,), eps=None, elementwise_affine=True)
      (feed_forward): SwiGLU(
        (gate_projection): Linear(in_features=32, out_features=96, bias=False)
        (up_projection): Linear(in_features=32, out_features=96, bias=False)
        (down_projection): Linear(in_features=96, out_features=32, bias=False)
      )
    )
  )
  (final_norm): RMSNorm((32,), eps=None, elementwise_affine=True)
  (lm_head): Linear(in_features

In [34]:
full_ids: torch.Tensor = torch.tensor([[5, 8, 2, 9]])

prompt_ids: torch.Tensor = full_ids[:, :-1]
new_token_ids: torch.Tensor = full_ids[:, -1:]

print("full:", full_ids.shape)
print("prompt:", prompt_ids.shape)
print("new token:", new_token_ids.shape)

full: torch.Size([1, 4])
prompt: torch.Size([1, 3])
new token: torch.Size([1, 1])


In [35]:
with torch.no_grad():
    full_result = model(full_ids)

if not isinstance(
    full_result,
    torch.Tensor,
):
    raise RuntimeError("Expected ordinary model output.")

full_logits: torch.Tensor = full_result

print(full_logits.shape)
full_last_logits: torch.Tensor = full_logits[:, -1:, :]

print(full_last_logits.shape)

torch.Size([1, 4, 100])
torch.Size([1, 1, 100])


In [36]:
# Prefill

with torch.no_grad():
    prefill_result = model(
        prompt_ids,
        use_cache=True,
    )

if not isinstance(
    prefill_result,
    tuple,
):
    raise RuntimeError("Expected cached model output.")

prompt_logits, cache = prefill_result
print(
    "prompt logits:",
    prompt_logits.shape,
)

print(
    "cached layers:",
    len(cache),
)

prompt logits: torch.Size([1, 3, 100])
cached layers: 3


In [37]:
for layer_index, (
    key_cache,
    value_cache,
) in enumerate(cache):
    print(
        f"layer {layer_index}:",
        "K",
        key_cache.shape,
        "V",
        value_cache.shape,
    )

layer 0: K torch.Size([1, 4, 3, 8]) V torch.Size([1, 4, 3, 8])
layer 1: K torch.Size([1, 4, 3, 8]) V torch.Size([1, 4, 3, 8])
layer 2: K torch.Size([1, 4, 3, 8]) V torch.Size([1, 4, 3, 8])


In [38]:
with torch.no_grad():
    decode_result = model(
        new_token_ids,
        past_key_values=cache,
        use_cache=True,
    )

if not isinstance(
    decode_result,
    tuple,
):
    raise RuntimeError("Expected cached model output.")

cached_logits, updated_cache = decode_result
print(
    "cached logits:",
    cached_logits.shape,
)

cached logits: torch.Size([1, 1, 100])


In [39]:
# Update Cache from 3 to 4.
for layer_index, (
    key_cache,
    value_cache,
) in enumerate(updated_cache):
    print(
        f"layer {layer_index}:",
        "K",
        key_cache.shape,
        "V",
        value_cache.shape,
    )

layer 0: K torch.Size([1, 4, 4, 8]) V torch.Size([1, 4, 4, 8])
layer 1: K torch.Size([1, 4, 4, 8]) V torch.Size([1, 4, 4, 8])
layer 2: K torch.Size([1, 4, 4, 8]) V torch.Size([1, 4, 4, 8])


In [40]:
logits_match: bool = torch.allclose(
    full_last_logits,
    cached_logits,
    atol=1e-5,
)

print(
    "cached logits match full logits:",
    logits_match,
)

cached logits match full logits: True


In [41]:
maximum_difference: float = (
    (full_last_logits - cached_logits).abs().max().item()
)

print(
    "maximum absolute difference:",
    maximum_difference,
)

maximum absolute difference: 3.5762786865234375e-07


### What This Test Verifies

Matching final logits verify the complete cached inference path:

- token embeddings,
- one independent KV cache per Transformer layer,
- RoPE position offsets,
- cached attention,
- residual connections,
- SwiGLU transformations,
- final RMS normalization,
- and the language-model head.

Therefore,

$$
\boxed{
\text{cached decoding}
\approx
\text{full-sequence decoding}
}
$$

up to floating-point error.

The KV cache is an inference optimization, not a different language
model.

## 12. Cached Autoregressive Generation

The KV cache is now mathematically verified.

The final step is to use it inside an autoregressive generation loop.

Generation has two phases:

### Prefill

The complete prompt is processed once:

$$
(B,T)
\rightarrow
\text{GPT}
\rightarrow
(B,T,V).
$$

At the same time, every Transformer layer creates its initial KV cache.

### Decode

After the prompt, each generation step processes only one new token:

$$
(B,1)
\rightarrow
\text{GPT + KV cache}
\rightarrow
(B,1,V).
$$

The newly computed keys and values are appended to the cache after every
step.

The generation process remains sequential, but previously processed
tokens are no longer recomputed.

In [42]:
def generate_greedy_cached(
    model: GPT, input_ids: torch.Tensor, max_new_tokens: int
) -> torch.Tensor:
    """Generate tokens greedily using a KV cache.

    Args:
        model: Decoder-only Transformer supporting KV caching.
        input_ids: Prompt token IDs with shape `(B, T)`.
        max_new_tokens: Number of new tokens to generate.

    Returns:
        Token IDs containing the original prompt followed by generated
        tokens.
    """
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens must be non-negative")

    generated: torch.Tensor = input_ids

    model.eval()

    with torch.no_grad():
        # Prefill processes the complete prompt once and creates one
        # KV-cache pair for every Transformer layer.
        prefill_result = model(input_ids, use_cache=True)

        if not isinstance(prefill_result, tuple):
            raise RuntimeError("Expected cached model output.")

        logits, cache = prefill_result

        for _ in range(max_new_tokens):
            # The final prompt position predicts the next token.
            next_token_logits: torch.Tensor = logits[:, -1, :]

            next_token: torch.Tensor = torch.argmax(
                next_token_logits, dim=-1, keepdim=True
            )  # (B, T, C) -> dim=-1 => C

            generated = torch.cat(
                [generated, next_token], dim=1
            )  # (B, T, C) -> dim=1 => T

            decode_result = model(
                next_token, past_key_values=cache, use_cache=True
            )

            if not isinstance(decode_result, tuple):
                raise RuntimeError("Expected cached model output.")
            logits, cache = decode_result

    return generated

In [43]:
prompt_ids: torch.Tensor = torch.tensor([[5, 8, 2, 9]])

naive_generated: torch.Tensor = generate_greedy(
    model=model,
    input_ids=prompt_ids,
    max_new_tokens=8,
)

cached_generated: torch.Tensor = generate_greedy_cached(
    model=model,
    input_ids=prompt_ids,
    max_new_tokens=8,
)

print("naive:")
print(naive_generated)

print("\ncached:")
print(cached_generated)

naive:
tensor([[ 5,  8,  2,  9, 36, 38,  9, 27, 92, 36, 38,  9]])

cached:
tensor([[ 5,  8,  2,  9, 36, 38,  9, 27, 92, 36, 38,  9]])


In [44]:
print(
    "same generated sequence:",
    torch.equal(
        naive_generated,
        cached_generated,
    ),
)

same generated sequence: True


## 13. KV-Cache Memory Cost

A KV cache reduces repeated computation, but it requires memory.

For one Transformer layer, the cached keys have shape

$$
(B,H,T,D),
$$

and the cached values have the same shape:

$$
(B,H,T,D).
$$

Therefore, one layer stores

$$
2BHTD
$$

scalar values.

Because standard multi-head attention uses

$$
C = HD,
$$

this can also be written as

$$
2BTC.
$$

For a model with $L$ Transformer layers, the total KV-cache size is

$$
\boxed{
2LBHTD
}
$$

or equivalently

$$
\boxed{
2LBTC
}.
$$

The factor of two comes from storing both keys and values.

In [45]:
def kv_cache_memory_bytes(
    batch_size: int,
    num_layers: int,
    num_heads: int,
    sequence_length: int,
    head_dim: int,
    bytes_per_element: int,
) -> int:
    """Estimate KV-cache memory usage in bytes.

    Args:
        batch_size: Number of sequences in the batch.
        num_layers: Number of Transformer layers.
        num_heads: Number of key/value heads.
        sequence_length: Number of cached token positions.
        head_dim: Feature dimension of each attention head.
        bytes_per_element: Storage size of one scalar value.

    Returns:
        Estimated total memory used by cached keys and values.
    """
    key_elements: int = (
        batch_size * num_layers * num_heads * sequence_length * head_dim
    )

    # Keys and values have identical shapes.
    kv_elements: int = 2 * key_elements

    return kv_elements * bytes_per_element

In [46]:
memory_bytes: int = kv_cache_memory_bytes(
    batch_size=1,
    num_layers=3,
    num_heads=4,
    sequence_length=100,
    head_dim=8,
    bytes_per_element=2,
)

print("bytes:", memory_bytes)
print("KiB:", memory_bytes / 1024)

bytes: 38400
KiB: 37.5


## Takeaways

Autoregressive generation differs fundamentally from training.

During training, all target positions are already known, so a causal
Transformer can process the sequence in parallel.

During generation, each new token depends on the previously generated
tokens:

$$
x_{t+1}
\sim
p(x_{t+1}\mid x_{\le t}).
$$

Generation is therefore sequential across newly produced tokens.

### Sampling controls how the next token is selected

Greedy decoding always selects the highest-logit token.

Temperature rescales logits:

$$
z
\rightarrow
\frac{z}{\tau}.
$$

Lower temperatures sharpen the distribution, while higher temperatures
flatten it.

Top-k sampling restricts generation to a fixed number of high-logit
tokens.

Top-p sampling instead keeps the smallest set of tokens whose cumulative
probability reaches a chosen threshold.

These techniques change token selection, not the underlying language
model.

### Naive generation repeats unnecessary computation

Without caching, every decoding step processes the complete sequence
again:

<pre>
A B C
A B C D
A B C D E
A B C D E F
</pre>

However, causal attention guarantees that future tokens cannot modify
representations at earlier positions.

Previously computed keys and values can therefore be reused.

### KV caching separates inference into two phases

During **prefill**, the complete prompt is processed once:

$$
(B,T)
\rightarrow
(B,T,C).
$$

Each Transformer layer stores its keys and values.

During **decode**, only one newly generated token needs to enter the
model:

$$
(B,1)
\rightarrow
(B,1,C).
$$

For each layer, the new query has shape

$$
(B,H,1,D),
$$

while the cached keys and values have shape

$$
(B,H,T_{\text{cache}},D).
$$

The current query therefore attends over the complete history without
recomputing previous key and value projections.

### Every Transformer layer owns an independent cache

A model with $L$ layers stores

$$
[
(K^{(0)},V^{(0)}),
\ldots,
(K^{(L-1)},V^{(L-1)})
].
$$

RoPE must use the cache length as a position offset so that newly decoded
tokens retain their correct absolute positions.

### Caching does not change model semantics

A correct implementation satisfies

$$
\text{full-sequence logits}
\approx
\text{cached-decoding logits}
$$

up to floating-point error.

The KV cache is therefore an inference optimization, not a different
attention algorithm.

### KV caching trades memory for computation

For standard multi-head attention, the total KV-cache storage across
$L$ layers is

$$
2LBHTD
$$

scalar values.

Since

$$
C=HD,
$$

this is equivalently

$$
2LBTC.
$$

KV-cache memory therefore grows linearly with context length.

This creates the motivation for the next architectural question:

> Do all query heads really need their own key and value heads?

That question leads directly to Multi-Query Attention and
Grouped-Query Attention.